In [1]:
import sys
import os
sys.path.insert(0, os.path.abspath('/home/ElasticNotebook'))
%load_ext ElasticNotebook
from elastic.core.common.pandas import compare_df
import pickle
import cudf

Enabled rmm statistics


In [2]:
%load_ext cudf.pandas

/opt/conda/envs/rapids-25.02/lib/python3.10/site-packages/cudf/pandas/__init__.py:65: UserWarning: cudf.pandas detected an already configured memory resource, ignoring 'CUDF_PANDAS_RMM_MODE'=managed_pool
  warnings.warn(


In [3]:
%LoadCheckpoint /home/dias-benchmarks/notebooks/erikbruin/nlp-on-student-writing-eda/src/rewritten/cell_19/checkpoints/post_cell_20_try_5.pickle

trying: ['i_3', 'i_1']
me:  13
me:  11
trying: ['i_3', 'i_1']
me:  13
me:  11
trying: ['last_ones']
me:  20
trying: ['tqdm']
me:  0
trying: ['sample_submission']
me:  2
trying: ['myword']
me:  12
trying: ['train_first_last']
me:  11
trying: ['len_dict']
me:  12
trying: ['cols_to_display']
me:  14
trying: ['word_dict']
me:  12
trying: ['counter']
me:  11
trying: ['glob']
me:  0
trying: ['mylen']
me:  12
trying: ['ax']
me:  11
trying: ['IREWR_plug_2']
me:  16
trying: ['train_first']
me:  11
trying: ['train_last']
me:  11
trying: ['train']
me:  14
trying: ['stopwords']
me:  1
trying: ['plt']
me:  0
trying: ['add_gap_rows_2']
me:  22
trying: ['style']
me:  0
trying: ['CountVectorizer']
me:  0
trying: ['sample_discourse_id']
me:  2
trying: ['FuncFormatter']
me:  0
trying: ['warnings']
me:  0
trying: ['pd']
me:  0
trying: ['cols_to_merge']
me:  13
trying: ['ax2']
me:  8
trying: ['ax1']
me:  8
trying: ['np']
me:  0
trying: ['orig_output']
me:  21
trying: ['test_txt']
me:  1
trying: ['av_per_e

In [4]:
%%RecordEvent
%%time
### cell 21 ###

# lowercase the discourse text (runs on GPU via the cudf.pandas extension)
train['discourse_text'] = train['discourse_text'].str.lower()

# build the stopword list
stop_english = stopwords.words('english')
other_words_to_take_out = ['school', 'students', 'people', 'would', 'could', 'many']
stop_english.extend(other_words_to_take_out)

# split each discourse_text into words and explode so each word gets its own row
exploded = train.assign(Word=train['discourse_text'].str.split()).explode('Word')

# filter out stopwords (runs on GPU)
filtered = exploded[~exploded['Word'].isin(stop_english)]

# count frequencies of each word by discourse_type
freq = (
    filtered
    .groupby(['discourse_type', 'Word'])
    .size()
    .reset_index(name='Frequency')
)

# for each discourse_type, take the top 10 words by Frequency
top10 = (
    freq
    .sort_values(['discourse_type', 'Frequency'], ascending=[True, False])
    .groupby('discourse_type')
    .head(10)
)

# build counts_dict as a plain dict mapping each discourse_type to its small DataFrame
counts_dict = {
    dt: (
        group
        .drop('discourse_type', axis=1)
        .set_index('Word')
        .sort_values('Frequency', ascending=True)
    )
    for dt, group in top10.groupby('discourse_type')
}

# extract the keys and final df as in the original
keys = list(counts_dict.keys())
t_last = keys[-1]
df = train[train['discourse_type'] == t_last]

CPU times: user 139 ms, sys: 43.9 ms, total: 183 ms
Wall time: 183 ms


In [5]:
%Checkpoint /home/dias-benchmarks/notebooks/erikbruin/nlp-on-student-writing-eda/src/rewritten/cell_19/checkpoints/post_cell_21_try_5.pickle

migration speed (bps): 787161180.9619622
---------------------------
variables to migrate:
total_gaps 48
stop_english 1912
IREWR_tmp 48
filtered 48
all_gaps 48
freq 48
test_txt 120
add_gap_rows_2 144
counts_dict 696
df 48
train 48
glob 144
tqdm 1072
style 72
i_3 28
CountVectorizer 1072
cols_to_merge 120
plt 72
FuncFormatter 1072
warnings 72
sample_discourse_id 32
train_first_last 48
myword 28
i_1 28
len_dict 589920
counter 28
word_dict 589920
ax 48
mylen 28
train_first 48
myid 61
train_last 48
data 2813
ax1 48
txt_file 208
t 166
ax2 48
orig_output 48
sample_submission 48
train_txt 126104
pd 72
IREWR_plug_2 61
cols_to_display 120
t_last 57
top10 48
last_ones 48
keys 120
IREWR_plug_1 61
av_per_essay 48
other_words_to_take_out 104
add_gap_rows 144
stopwords 48
IREWR_tmp2 48
exploded 48
np 72
---------------------------
variables to recompute:
[]
---------------------------
cells to recompute:
[]
Checkpoint saved to: /home/dias-benchmarks/notebooks/erikbruin/nlp-on-student-writing-eda/src/

In [6]:
%PrintCellInfo opt_cell_exec_info

======= Cell 0 =======
Input variables []
Active variables ['train_txt', 'stopwords', 'sample_submission', 'train']
Intermediate variables ['test_txt']
Future variables []
Modified dataframes
======= Cell 1 =======
Input variables []
Active variables ['train', 'sample_discourse_id']
Intermediate variables ['sample_submission']
Future variables ['stopwords', 'train_txt']
Modified dataframes
  train
    Input columns: set()
    Changed columns: {'discourse_id', 'discourse_type_num', 'predictionstring', 'discourse_type', 'id', 'discourse_end', 'discourse_text', 'discourse_start'}
    Created columns: set()
    Deleted columns: set()
  sample_submission
    Input columns: set()
    Changed columns: {'predictionstring', 'id', 'class'}
    Created columns: set()
    Deleted columns: set()
======= Cell 2 =======
Input variables []
Active variables ['train', 'cols_to_display']
Intermediate variables []
Future variables ['stopwords', 'sample_discourse_id', 'train_txt']
Modified dataframes
  tra

In [7]:

with open("/home/dias-benchmarks/notebooks/erikbruin/nlp-on-student-writing-eda/src/opt_cell_exec_info_21_try_5.pkl", "wb") as f:
    pickle.dump(opt_cell_exec_info[21], f)


In [8]:
opt_output = Out.get(4)

In [9]:
df_opt = df
counts_dict_opt = counts_dict
train_opt = train
%LoadCheckpoint /home/dias-benchmarks/notebooks/erikbruin/nlp-on-student-writing-eda/src/annotated/checkpoints/post_cell_21.pickle
assert compare_df(df_opt, df)
assert counts_dict_opt == counts_dict
assert compare_df(train_opt, train)

import numpy as np
from elastic.core.common.pandas import is_type_styler
is_orig_output_pd = isinstance(orig_output, (pd.Series, pd.DataFrame, pd.Index))
is_opt_output_pd = isinstance(opt_output, (pd.Series, pd.DataFrame, pd.Index))
is_orig_output_array = isinstance(orig_output, (cudf.pandas._wrappers.numpy.ndarray, np.ndarray))
is_opt_output_array = isinstance(opt_output, (cudf.pandas._wrappers.numpy.ndarray, np.ndarray))
is_orig_output_styler = is_type_styler(type(orig_output))
is_opt_output_styler = is_type_styler(type(opt_output))
if is_orig_output_styler and is_opt_output_styler:
    assert orig_output.to_html() == opt_output.to_html()
elif is_orig_output_styler:
    assert orig_output.to_html() == opt_output.to_html()
elif is_opt_output_styler:
    assert opt_output.to_html() == orig_output

if is_orig_output_pd and is_opt_output_pd:
    assert orig_output.equals(opt_output)
# TODO(jie): this is a hack.
elif ((is_orig_output_pd or is_opt_output_pd) and (is_orig_output_array or is_opt_output_array)) or (is_orig_output_array and is_opt_output_array):
    assert list(orig_output) == list(opt_output)
else:
    assert orig_output == opt_output


trying: ['i_1', 'i_3']
me:  11
me:  13
trying: ['i_1', 'i_3']
me:  11
me:  13
trying: ['test_txt']
me:  1
trying: ['train_txt']
me:  1
trying: ['pd']
me:  0
trying: ['glob']
me:  0
trying: ['np']
me:  0
trying: ['text']
me:  24
trying: ['sample_discourse_id']
me:  2
trying: ['myword']
me:  12
trying: ['dt']
me:  24
trying: ['len_dict']
me:  12
trying: ['word_dict']
me:  12
trying: ['mylen']
me:  12
trying: ['cols_to_display']
me:  14
trying: ['myid']
me:  12
trying: ['df1']
me:  24
trying: ['style']
me:  0
trying: ['data']
me:  12
trying: ['CountVectorizer']
me:  0
trying: ['txt_file']
me:  12
trying: ['cols_to_merge']
me:  13
trying: ['ax1']
me:  8
trying: ['plt']
me:  0
trying: ['t']
me:  12
trying: ['train']
me:  24
trying: ['FuncFormatter']
me:  0
trying: ['sample_submission']
me:  2
trying: ['ax2']
me:  8
trying: ['warnings']
me:  0
trying: ['IREWR_tmp2']
me:  24
trying: ['total_gaps']
me:  24
trying: ['last_ones']
me:  24
trying: ['train_first_last']
me:  11
trying: ['stopwords']

ValueError: The truth value of a DataFrame is ambiguous. Use a.empty, a.bool(), a.item(), a.any() or a.all().